In [ ]:
import pandas as pd

golden = pd.read_csv("golden_set_200_reviewed.csv")

print("Rows:", len(golden))
print("Unique tweet IDs:", golden["tweet_id"].nunique())

print("\nColumns:")
print(golden.columns.tolist())

print("\nMissing values:")
print(golden[
    ["final_intent",
     "final_multi_issue",
     "final_context_required",
     "reviewed"]
].isna().sum())

print("\nReviewed:")
print(golden["reviewed"].value_counts(dropna=False))

print("\nIntent distribution:")
print(golden["final_intent"].value_counts())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Most common intent in the golden set
majority_intent = golden["final_intent"].mode()[0]

# Predict the same intent for every example
golden["baseline_1_prediction"] = majority_intent

accuracy = accuracy_score(
    golden["final_intent"],
    golden["baseline_1_prediction"]
)

print("Majority baseline intent:", majority_intent)
print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(
    classification_report(
        golden["final_intent"],
        golden["baseline_1_prediction"],
        zero_division=0
    )
)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Use the human-reviewed golden set
X = golden["text"].fillna("")
y = golden["final_intent"]

# Split golden data for training/testing the simple baseline
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# TF-IDF + Logistic Regression
tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=10000
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Train
tfidf_lr.fit(X_train, y_train)

# Predict
y_pred = tfidf_lr.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print("TF-IDF + Logistic Regression")
print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

In [ ]:
import pandas as pd

# Load development data
dev = pd.read_csv("development_set.csv")

# Load final human-reviewed golden set
golden = pd.read_csv("golden_set_200_reviewed.csv")

print("Development rows:", len(dev))
print("Golden rows:", len(golden))

In [ ]:
import pandas as pd

dev = pd.read_csv("development_set.csv")
test = pd.read_csv("test_set.csv")
golden = pd.read_csv("golden_set_200_reviewed.csv")

print("=== FILE SIZES ===")
print("Development:", len(dev))
print("Test:", len(test))
print("Golden:", len(golden))

print("\n=== DEVELOPMENT COLUMNS ===")
print(dev.columns.tolist())

print("\n=== TEST COLUMNS ===")
print(test.columns.tolist())

print("\n=== GOLDEN COLUMNS ===")
print(golden.columns.tolist())

# Check golden overlap with development/test
golden_ids = set(golden["tweet_id"])
dev_ids = set(dev["tweet_id"])
test_ids = set(test["tweet_id"])

print("\n=== OVERLAP CHECK ===")
print("Golden ∩ Development:", len(golden_ids & dev_ids))
print("Golden ∩ Test:", len(golden_ids & test_ids))
print("Development ∩ Test:", len(dev_ids & test_ids))

In [ ]:
import re
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

def rule_based_intent(text):
    text = str(text).lower()

    # Context-dependent / reaction messages
    context_words = [
        "thanks", "thank you", "thx", "okay", "ok", "yes",
        "that worked", "works now", "done", "i tried that",
        "still waiting", "finally", "sorry for delay"
    ]

    if (
        len(text.strip()) < 20
        or any(word in text for word in context_words)
        and not any(issue in text for issue in [
            "password", "wifi", "battery", "update", "charging",
            "screen", "payment", "music", "login", "keyboard"
        ])
    ):
        return "CONTEXT_REQUIRED"

    # Account / login
    if re.search(
        r"\b(apple id|icloud password|password|login|log in|sign in|signin|activation lock|account)\b",
        text
    ):
        return "ACCOUNT_LOGIN"

    # Billing / purchase
    if re.search(
        r"\b(payment|paying|paid|charge|charged|billing|subscription|purchase|purchased|order|preorder|refund)\b",
        text
    ):
        return "PURCHASE_BILLING"

    # Battery / charging
    if re.search(
        r"\b(battery|battery drain|draining|charging|charger|overheat|overheating|hot)\b",
        text
    ):
        return "BATTERY_CHARGING"

    # iOS / software update
    if re.search(
        r"\b(ios update|ios 1[0-9]|ios 11|ios 12|ios 13|ios 14|ios 15|ios 16|ios 17|ios 18|updated|update|updating|downgrade)\b",
        text
    ):
        return "IOS_UPDATE_ISSUE"

    # Network
    if re.search(
        r"\b(wifi|wi-fi|internet|bluetooth|mobile data|cellular|network|signal|connection|connect|connecting)\b",
        text
    ):
        return "NETWORK_CONNECTIVITY"

    # Hardware / repair
    if re.search(
        r"\b(broken|screen|display|button|home button|speaker|earphones|headphones|hardware|repair|replacement|replace|warranty|damaged|damage)\b",
        text
    ):
        return "HARDWARE_REPAIR"

    # Apple services
    if re.search(
        r"\b(apple music|itunes|apple pay|app store|icloud sync|syncing|apple service)\b",
        text
    ):
        return "APPLE_SERVICE_ISSUE"

    # Device performance
    if re.search(
        r"\b(slow|lag|lagging|freeze|freezing|frozen|crash|crashing|crashes|performance)\b",
        text
    ):
        return "DEVICE_PERFORMANCE"

    # Feature / settings
    if re.search(
        r"\b(keyboard|camera|setting|settings|notification|airplay|storage|app|apps|feature|volume|music tab|lightning|autocorrect)\b",
        text
    ):
        return "FEATURE_SETTINGS"

    return "UNKNOWN_ESCALATE"


# Run on the 200 human-reviewed golden examples
golden["rule_prediction"] = golden["text"].fillna("").apply(rule_based_intent)

# Metrics
y_true = golden["final_intent"]
y_pred = golden["rule_prediction"]

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("=== BASELINE #2: RULE-BASED CLASSIFIER ===")
print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    zero_division=0
))

In [ ]:
baseline_results = golden[
    ["tweet_id", "text", "final_intent",
     "final_multi_issue", "final_context_required"]
].copy()

baseline_results["rule_prediction"] = golden["rule_prediction"]

baseline_results.to_csv(
    "baseline_rule_predictions.csv",
    index=False
)

print("Saved: baseline_rule_predictions.csv")

In [ ]:
import pandas as pd

apple = pd.read_csv("apple_support.csv")

print("Rows:", len(apple))
print("Columns:", apple.columns.tolist())

print("\nInbound:")
print(apple["inbound"].value_counts())

print("\nCompany replies:")
print(apple[apple["inbound"] == False][[
    "tweet_id",
    "text",
    "in_response_to_tweet_id"
]].head(10))

In [ ]:
import pandas as pd

# Load full AppleSupport dataset
apple = pd.read_csv("apple_support.csv")

# Customer and company tweets
customers = apple[apple["inbound"] == True].copy()
support = apple[apple["inbound"] == False].copy()

# Match:
# customer tweet_id  <--  support tweet's in_response_to_tweet_id
pairs = customers.merge(
    support[
        ["tweet_id", "text", "in_response_to_tweet_id"]
    ],
    left_on="tweet_id",
    right_on="in_response_to_tweet_id",
    how="inner",
    suffixes=("_customer", "_support")
)

# Keep only useful columns
historical_pairs = pairs[[
    "tweet_id_customer",
    "text_customer",
    "tweet_id_support",
    "text_support"
]].rename(columns={
    "tweet_id_customer": "customer_tweet_id",
    "text_customer": "customer_text",
    "tweet_id_support": "support_tweet_id",
    "text_support": "support_reply"
})

print("Historical customer → support pairs:", len(historical_pairs))

print("\nSample pairs:")
display(historical_pairs.head(10))

In [ ]:
historical_pairs.to_csv(
    "historical_support_pairs.csv",
    index=False
)

print("Saved: historical_support_pairs.csv")

show the 5 most similar AppleSupport cases + their real replies.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

historical_pairs = pd.read_csv("historical_support_pairs.csv")

retrieval_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=50000,
    stop_words="english"
)

customer_matrix = retrieval_vectorizer.fit_transform(
    historical_pairs["customer_text"].fillna("")
)

print("Historical pairs:", len(historical_pairs))
print("TF-IDF matrix shape:", customer_matrix.shape)

In [ ]:
def retrieve_similar_cases(query, top_k=5):
    query_vector = retrieval_vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        customer_matrix
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = historical_pairs.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]

    return results[
        [
            "customer_tweet_id",
            "customer_text",
            "support_tweet_id",
            "support_reply",
            "similarity"
        ]
    ]

In [ ]:
query = golden.iloc[0]["text"]

results = retrieve_similar_cases(query, top_k=5)

display(results)

And the first result has similarity 1.0 because the test query itself exists in the historical pair dataset. That's a problem for our final evaluation: we must prevent the agent from retrieving the exact same tweet when evaluating the 200 golden examples




**fix retrieval leakage**

In [ ]:
def retrieve_similar_cases(query, top_k=5, exclude_tweet_id=None):
    query_vector = retrieval_vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        customer_matrix
    )[0]

    # Exclude the exact same customer tweet
    if exclude_tweet_id is not None:
        mask = (
            historical_pairs["customer_tweet_id"].astype(str)
            == str(exclude_tweet_id)
        )
        similarities[mask.values] = -1

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = historical_pairs.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]

    return results[
        [
            "customer_tweet_id",
            "customer_text",
            "support_tweet_id",
            "support_reply",
            "similarity"
        ]
    ]

In [ ]:
query = golden.iloc[0]["text"]
tweet_id = golden.iloc[0]["tweet_id"]

results = retrieve_similar_cases(
    query,
    top_k=5,
    exclude_tweet_id=tweet_id
)

display(results)

# Lets do **classifier that the agent will us**e

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Training data = human-reviewed golden examples
X_train = golden["text"].fillna("")
y_train = golden["final_intent"]

intent_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            max_features=20000,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

intent_model.fit(X_train, y_train)

print("Intent classifier trained successfully.")

**agent's intent function**

In [ ]:
def predict_intent(text):
    prediction = intent_model.predict([str(text)])[0]
    probabilities = intent_model.predict_proba([str(text)])[0]

    confidence = probabilities.max()

    return prediction, confidence

In [ ]:
query = golden.iloc[0]["text"]

intent, confidence = predict_intent(query)

print("Tweet:", query)
print("Predicted intent:", intent)
print("Confidence:", round(confidence, 4))

classifier predicted the right intent:

IOS_UPDATE_ISSUE

but confidence is only: 0.1734
 >That's very low. The reason is that we're training on only 200 examples, while there are 11 intent classes. That's not enough for a reliable classifie




### Next Approach

We will not use this classifier as the main agent for now.

Instead, we will use a better approach:

**Customer Message**
↓
**LLM understands the customer's intent**
+
**Historical retrieval finds similar support cases**
↓
**Generate a grounded reply**
↓
**Decide: Auto-handle or Escalate to a human**

The **200 human-reviewed examples** will remain our evaluation set. We will use them to measure how well the system performs, rather than using them as the main training data.

We already have a valuable historical dataset:

**78,440 customer → AppleSupport reply pairs**

Now, we can focus on building the **reply-generation and escalation logic** using this data.


**let's now build the reply-generation + escalation logic,**

In [ ]:
import pandas as pd

# Quick retrieval quality check on 10 golden examples
for i in range(10):
    row = golden.iloc[i]

    results = retrieve_similar_cases(
        row["text"],
        top_k=3,
        exclude_tweet_id=row["tweet_id"]
    )

    print("=" * 80)
    print("Golden tweet:", row["text"])
    print("Human intent:", row["final_intent"])
    print("\nRetrieved cases:")

    for _, r in results.iterrows():
        print("\nSimilarity:", round(r["similarity"], 3))
        print("Customer:", r["customer_text"])
        print("AppleSupport:", r["support_reply"])

**it finds highly similar real AppleSupport cases and the actual support replies.**

But before we build the reply generator, there’s one important thing we should fix.

The retrieval corpus should use only `development_set.csv`, not the full `apple_support.csv`.

If we use the full dataset, some conversations from the golden evaluation set could also appear in retrieval. That would make the system look better during evaluation than it actually is.

So, to keep the evaluation fair, we’ll use `development_set.csv` for retrieval and keep the golden set completely separate.


**Build clean historical pairs from development_set.csv**

In [ ]:
import pandas as pd

# Make sure development dataframe exists
# If your variable is called dev instead, use dev here.
development = pd.read_csv("development_set.csv")

# IDs that are allowed for retrieval/training
dev_customer_ids = set(
    development["tweet_id"].astype(str)
)


dev_customer_ids = set(
    development["tweet_id"].astype(str)
)

# Use the original full AppleSupport data to find the replies
customers = development.copy()
support = apple[apple["inbound"] == False].copy()

# Match development customer tweets to AppleSupport replies
dev_pairs = customers.merge(
    support[
        ["tweet_id", "text", "in_response_to_tweet_id"]
    ],
    left_on="tweet_id",
    right_on="in_response_to_tweet_id",
    how="inner",
    suffixes=("_customer", "_support")
)

historical_pairs_dev = dev_pairs[[
    "tweet_id_customer",
    "text_customer",
    "tweet_id_support",
    "text_support"
]].rename(columns={
    "tweet_id_customer": "customer_tweet_id",
    "text_customer": "customer_text",
    "tweet_id_support": "support_tweet_id",
    "text_support": "support_reply"
})

print("Development customer tweets:", len(development))
print("Clean historical pairs:", len(historical_pairs_dev))

print(
    "\nGolden IDs found in retrieval corpus:",
    len(
        set(historical_pairs_dev["customer_tweet_id"].astype(str))
        & set(golden["tweet_id"].astype(str))
    )
)

display(historical_pairs_dev.head())

In [ ]:
print("Clean retrieval pairs:", len(historical_pairs_dev))
print(historical_pairs_dev.columns.tolist())

historical_pairs_dev.to_csv(
    "historical_support_pairs_dev.csv",
    index=False
)

print("Saved successfully.")

In [ ]:
def generate_support_reply(customer_message, top_k=3):
    # 1. Predict intent
    predicted_intent, intent_confidence = predict_intent(customer_message)

    # 2. Retrieve similar cases (excluding the query tweet if it's from golden set)
    # For a general customer message, we don't have a tweet_id to exclude.
    retrieved_cases = retrieve_similar_cases(customer_message, top_k=top_k)

    # 3. Simple reply generation logic (placeholder for more advanced LLM grounding)
    reply_message = f"Predicted Intent: {predicted_intent} (Confidence: {intent_confidence:.2f})\n\n"
    reply_message += "Here are some similar historical support interactions that might help:\n"

    for idx, row in retrieved_cases.iterrows():
        reply_message += f"\nSimilarity: {row['similarity']:.3f}\n"
        reply_message += f"Customer: {row['customer_text']}\n"
        reply_message += f"AppleSupport: {row['support_reply']}\n"

    return reply_message

# Example usage with a sample customer message
sample_customer_message = golden.iloc[0]["text"]

print(generate_support_reply(sample_customer_message))

In [ ]:
# Reload the files fresh so there is no accidental old variable
import pandas as pd

development = pd.read_csv("development_set.csv")
golden = pd.read_csv("golden_set_200_reviewed.csv")

dev_ids = set(development["tweet_id"].astype(str))
golden_ids = set(golden["tweet_id"].astype(str))

print("Development rows:", len(development))
print("Golden rows:", len(golden))
print("Golden IDs in development:", len(dev_ids & golden_ids))

rebuild the retrieval pairs from scratch

In [ ]:
# Full AppleSupport dataset
apple = pd.read_csv("apple_support.csv")

# Only development customer tweets
dev_customers = development[development["inbound"] == True].copy()

# AppleSupport replies
support_replies = apple[apple["inbound"] == False].copy()

# Customer -> direct AppleSupport reply
dev_pairs = dev_customers.merge(
    support_replies[
        ["tweet_id", "text", "in_response_to_tweet_id"]
    ],
    left_on="tweet_id",
    right_on="in_response_to_tweet_id",
    how="inner",
    suffixes=("_customer", "_support")
)

historical_pairs_dev = dev_pairs[[
    "tweet_id_customer",
    "text_customer",
    "tweet_id_support",
    "text_support"
]].rename(columns={
    "tweet_id_customer": "customer_tweet_id",
    "text_customer": "customer_text",
    "tweet_id_support": "support_tweet_id",
    "text_support": "support_reply"
})

# Final safety check
retrieval_ids = set(
    historical_pairs_dev["customer_tweet_id"].astype(str)
)

print("Historical pairs:", len(historical_pairs_dev))
print("Golden IDs in retrieval pairs:", len(retrieval_ids & golden_ids))

rebuild TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

retrieval_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=50000,
    stop_words="english"
)

customer_matrix = retrieval_vectorizer.fit_transform(
    historical_pairs_dev["customer_text"].fillna("")
)

print("TF-IDF matrix shape:", customer_matrix.shape)

retrieval function

In [ ]:
def retrieve_similar_cases(query, top_k=5, exclude_tweet_id=None):

    query_vector = retrieval_vectorizer.transform([str(query)])

    similarities = cosine_similarity(
        query_vector,
        customer_matrix
    )[0]

    if exclude_tweet_id is not None:
        mask = (
            historical_pairs_dev["customer_tweet_id"].astype(str)
            == str(exclude_tweet_id)
        )
        similarities[mask.values] = -1

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = historical_pairs_dev.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]

    return results[[
        "customer_tweet_id",
        "customer_text",
        "support_tweet_id",
        "support_reply",
        "similarity"
    ]]

In [ ]:
row = golden.iloc[0]

results = retrieve_similar_cases(
    row["text"],
    top_k=5,
    exclude_tweet_id=row["tweet_id"]
)

display(results)

Development rows: 78,157
Golden rows: 200
Golden IDs in development: 0
Historical pairs: 62,686
Golden IDs in retrieval pairs: 0

retrieved replies are mostly things like:

“Please DM us.”

“Let's look into this together.”

That is actually useful evidence. It tells us how AppleSupport historically handled these cases: ask for device/version details, troubleshoot, or move to DM.

Now we can build the real agent.

reply generation -
>We should use the retrieved examples to generate a response, but we must prevent the LLM from copying the historical tweet/reply blindly.

* predicted_intent
* draft_reply
* decision
* escalation_reason
* evidence

In [ ]:
# ============================================================
# Hiver AI Support Agent - Gemini + Historical Retrieval
# ============================================================

!pip -q install -U google-genai

import json
import re
import pandas as pd
from google import genai
from google.colab import userdata

# ------------------------------------------------------------
# 1. Load Gemini API key securely with robust fallback
# ------------------------------------------------------------

try:
    # Try to load standard named secret first
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    # Fall back directly to the raw key string instead of querying it as a secret name
    GOOGLE_API_KEY = "REDACTED_API_KEY"

if not GOOGLE_API_KEY:
    raise RuntimeError("GOOGLE_API_KEY is empty.")

try:
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("Gemini client configured successfully.")
except Exception as e:
    print(f"Could not configure Gemini Client: {e}. Falling back to simulation logic.")

# Current active Gemini model
MODEL_NAME = "gemini-3.6-flash"

# ------------------------------------------------------------
# 2. Intent taxonomy
# ------------------------------------------------------------

VALID_INTENTS = [
    "ACCOUNT_LOGIN",
    "APPLE_SERVICE_ISSUE",
    "PURCHASE_BILLING",
    "DEVICE_PERFORMANCE",
    "BATTERY_CHARGING",
    "IOS_UPDATE_ISSUE",
    "HARDWARE_REPAIR",
    "NETWORK_CONNECTIVITY",
    "FEATURE_SETTINGS",
    "CONTEXT_REQUIRED",
    "UNKNOWN_ESCALATE"
]

# ------------------------------------------------------------
# 3. Escalation policy (User Defined)
# ------------------------------------------------------------

def decide_escalation(
    predicted_intent,
    intent_confidence,
    retrieved_cases
):
    """
    Decide whether the request should be auto-handled or escalated.

    High-risk intents are always escalated.
    Low-risk intents can be auto-handled when:
    - intent confidence is reasonable
    - historical evidence is strong
    """

    # Always escalate sensitive / context-dependent cases
    always_escalate = {
        "ACCOUNT_LOGIN",
        "PURCHASE_BILLING",
        "HARDWARE_REPAIR",
        "CONTEXT_REQUIRED",
        "UNKNOWN_ESCALATE"
    }

    if predicted_intent in always_escalate:
        return (
            "ESCALATE",
            f"{predicted_intent} requires human support handling."
        )

    # No evidence = don't guess
    if retrieved_cases.empty:
        return (
            "ESCALATE",
            "No relevant historical AppleSupport evidence was found."
        )

    top_similarity = float(
        retrieved_cases["similarity"].max()
    )

    # Very weak retrieval evidence
    if top_similarity < 0.20:
        return (
            "ESCALATE",
            "Historical evidence is too weak to safely ground an automated reply."
        )

    # Low confidence + only moderate evidence
    if intent_confidence < 0.15 and top_similarity < 0.30:
        return (
            "ESCALATE",
            "Both intent confidence and historical evidence are weak."
        )

    # Otherwise allow the system to draft an answer
    return (
        "AUTO_HANDLE",
        "The intent is sufficiently supported by relevant historical cases."
    )

# ------------------------------------------------------------
# 4. Build historical evidence text
# ------------------------------------------------------------

def format_evidence(retrieved_cases):
    if retrieved_cases.empty:
        return "No historical evidence available."

    evidence = []
    for i, row in retrieved_cases.reset_index(drop=True).iterrows():
        evidence.append(
            f"""
CASE {i+1}
Similarity: {float(row['similarity']):.3f}

Customer:
{str(row['customer_text'])}

AppleSupport response:
{str(row['support_reply'])}
""".strip()
        )
    return "\n\n".join(evidence)

# ------------------------------------------------------------
# 5. Main support agent
# ------------------------------------------------------------

def generate_support_agent(message, tweet_id=None):
    message = str(message).strip()
    if not message:
        raise ValueError("Customer message is empty.")

    # ---- Intent prediction using existing classifier ----
    predicted_intent, intent_confidence = predict_intent(message)
    predicted_intent = str(predicted_intent)
    intent_confidence = float(intent_confidence)

    # Safety check
    if predicted_intent not in VALID_INTENTS:
        predicted_intent = "UNKNOWN_ESCALATE"

    # ---- Historical retrieval ----
    retrieved_cases = retrieve_similar_cases(
        message,
        top_k=5,
        exclude_tweet_id=tweet_id
    )

    # ---- Escalation policy ----
    decision, escalation_reason = decide_escalation(
        predicted_intent,
        intent_confidence,
        retrieved_cases
    )

    # ---- Evidence ----
    evidence_text = format_evidence(retrieved_cases)

    # --------------------------------------------------------
    # Gemini prompt
    # --------------------------------------------------------

    prompt = f"""
You are the AI customer-support assistant for a research prototype
based on historical AppleSupport Twitter conversations.

Your job is to produce a NEW support reply for the current customer.

CURRENT CUSTOMER MESSAGE:
{message}

INTERNAL INTENT PREDICTION:
{predicted_intent}

INTERNAL CONFIDENCE:
{intent_confidence:.4f}

ESCALATION POLICY DECISION:
{decision}

ESCALATION REASON:
{escalation_reason}

HISTORICAL APPLESUPPORT EVIDENCE:
{evidence_text}

IMPORTANT RULES:

1. Do not claim to be Apple.
2. Do not invent Apple policies, refunds, warranty coverage,
   guarantees, or troubleshooting steps.
3. Use the historical responses as evidence, not as text to copy.
4. Write a NEW reply suitable for a customer-support Twitter conversation.
5. Be concise and helpful.
6. Never include historical customer usernames or tweet IDs in the reply.
7. Never include shortened tracking URLs from historical tweets.
8. Do not mention that you are an AI.
9. If evidence is weak, do not pretend that a specific solution is known.
10. For ESCALATE, politely ask the customer to continue through DM
    or another support channel, consistent with the historical evidence.
11. Do not change the internally predicted intent.
12. The final decision must remain exactly:
    {decision}

Return ONLY valid JSON with exactly these keys:

{{
  "predicted_intent": "{predicted_intent}",
  "intent_confidence": {intent_confidence:.4f},
  "draft_reply": "string",
  "decision": "{decision}",
  "escalation_reason": "string",
  "evidence_used": ["short evidence reference 1", "short evidence reference 2"]
}}
"""

    # --------------------------------------------------------
    # Gemini call
    # --------------------------------------------------------
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config={
                "temperature": 0.2,
                "response_mime_type": "application/json"
            }
        )
        raw = response.text.strip()
        result = json.loads(raw)

        # Keep our internal values authoritative
        result["predicted_intent"] = predicted_intent
        result["intent_confidence"] = round(intent_confidence, 4)
        result["decision"] = decision
        result["escalation_reason"] = escalation_reason

        if not isinstance(result.get("evidence_used"), list):
            result["evidence_used"] = []

        result["draft_reply"] = str(result.get("draft_reply", "")).strip()
        return result

    except Exception as e:
        # Safe local fallback
        fallback_reply = (
            "We'd like to look into this with you. "
            "Please send us a direct message so we can gather more details "
            "and continue troubleshooting."
        )
        return {
            "predicted_intent": predicted_intent,
            "intent_confidence": round(intent_confidence, 4),
            "draft_reply": fallback_reply,
            "decision": "ESCALATE",
            "escalation_reason": (
                f"Gemini generation failed, so the system used the safe "
                f"fallback. Error: {type(e).__name__}"
            ),
            "evidence_used": []
        }

# ------------------------------------------------------------
# 6. Test on ONE golden example
# ------------------------------------------------------------

test_row = golden.iloc[0]
agent_output = generate_support_agent(
    message=test_row["text"],
    tweet_id=test_row["tweet_id"]
)

print("=== CUSTOMER ===")
print(test_row["text"])
print("\n=== HUMAN GOLD LABEL ===")
print(test_row["final_intent"])
print("\n=== AGENT OUTPUT ===")
print(json.dumps(agent_output, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# Gemini connection diagnostic
# ============================================================

from google import genai
from google.colab import userdata

try:
    # Try to load using the correct Secret name
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    # If the secret is not found, use the raw fallback key string directly
    GOOGLE_API_KEY = "REDACTED_API_KEY"

client = genai.Client(api_key=GOOGLE_API_KEY)

print("API key loaded:", bool(GOOGLE_API_KEY))

print("\nAvailable Gemini models that support generateContent:\n")

try:
    for m in client.models.list():
        actions = getattr(m, "supported_actions", None)

        if actions is None or "generateContent" in actions:
            print(m.name)

except Exception as e:
    print("MODEL LIST ERROR:")
    print(type(e).__name__)
    print(str(e))

Then test one model directly

In [ ]:
from google import genai
from google.colab import userdata

try:
    # Try to load the standard secret name
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    # Fall back directly to the raw key value instead of passing it to userdata.get()
    api_key = "REDACTED_API_KEY"

client = genai.Client(api_key=api_key)

# Updated to use the recommended active model version
MODEL_NAME = "gemini-3.6-flash"

try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents="Reply with exactly: GEMINI_TEST_OK"
    )

    print("Gemini test successful!")
    print(response.text)

except Exception as e:
    print("Gemini test failed")
    print("Type:", type(e).__name__)
    print("Error:", str(e))

valuate all 200 and save results

In [ ]:
import pandas as pd
import json
import time

# Make sure the results directory exists
import os
os.makedirs("results", exist_ok=True)

results = []

print("Starting evaluation on 200 human-reviewed golden examples...\n")

for i, row in golden.iterrows():

    tweet_id = row["tweet_id"]
    customer_text = row["text"]

    try:
        agent_output = generate_support_agent(
            message=customer_text,
            tweet_id=tweet_id
        )

        results.append({
            "tweet_id": tweet_id,
            "text": customer_text,
            "actual_intent": row["final_intent"],
            "actual_multi_issue": row["final_multi_issue"],
            "actual_context_required": row["final_context_required"],
            "predicted_intent": agent_output.get("predicted_intent"),
            "intent_confidence": agent_output.get("intent_confidence"),
            "draft_reply": agent_output.get("draft_reply"),
            "decision": agent_output.get("decision"),
            "escalation_reason": agent_output.get("escalation_reason"),
            "evidence_used": json.dumps(
                agent_output.get("evidence_used", []),
                ensure_ascii=False
            )
        })

    except Exception as e:

        results.append({
            "tweet_id": tweet_id,
            "text": customer_text,
            "actual_intent": row["final_intent"],
            "actual_multi_issue": row["final_multi_issue"],
            "actual_context_required": row["final_context_required"],
            "predicted_intent": "UNKNOWN_ESCALATE",
            "intent_confidence": 0.0,
            "draft_reply": "",
            "decision": "ESCALATE",
            "escalation_reason": f"Agent error: {type(e).__name__}",
            "evidence_used": "[]"
        })

    # Progress
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/200")

    # Small delay to reduce API pressure
    time.sleep(0.2)

evaluation_df = pd.DataFrame(results)

evaluation_df.to_csv(
    "results/golden_predictions.csv",
    index=False
)

print("\nEvaluation complete.")
print("Saved: results/golden_predictions.csv")
print("Rows:", len(evaluation_df))

calculate the intent results

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_true = evaluation_df["actual_intent"]
y_pred = evaluation_df["predicted_intent"]

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

print("=== FINAL INTENT RESULTS ===")
print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        zero_division=0
    )
)

confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

labels = sorted(golden["final_intent"].unique())

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels
)

fig, ax = plt.subplots(figsize=(12, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

disp.plot(
    ax=ax,
    xticks_rotation=45,
    values_format="d"
)

plt.title("Intent Confusion Matrix")
plt.tight_layout()
plt.show()

decision distribution

In [ ]:
print("=== ESCALATION / AUTO-HANDLE ===")

print(
    evaluation_df["decision"]
    .value_counts()
)

print("\nPercentages:")
print(
    evaluation_df["decision"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

compare predicted vs human intent

In [ ]:
evaluation_df["intent_correct"] = (
    evaluation_df["actual_intent"]
    == evaluation_df["predicted_intent"]
)

print("Correct intent predictions:")
print(
    evaluation_df["intent_correct"]
    .value_counts()
)

print(
    "\nIntent agreement:",
    round(
        evaluation_df["intent_correct"].mean() * 100,
        2
    ),
    "%"
)

In [ ]:
import json
from google import genai
from google.colab import userdata

try:
    # Try to load the standard secret name first
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    # Fall back directly to the raw key value
    api_key = "REDACTED_API_KEY"

client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.8-flash"

VALID_INTENTS = [
    "ACCOUNT_LOGIN",
    "APPLE_SERVICE_ISSUE",
    "PURCHASE_BILLING",
    "DEVICE_PERFORMANCE",
    "BATTERY_CHARGING",
    "IOS_UPDATE_ISSUE",
    "HARDWARE_REPAIR",
    "NETWORK_CONNECTIVITY",
    "FEATURE_SETTINGS",
    "CONTEXT_REQUIRED",
    "UNKNOWN_ESCALATE"
]

taxonomy_text = """
ACCOUNT_LOGIN:
Apple ID, iCloud login, password, sign-in, activation lock, account access.

APPLE_SERVICE_ISSUE:
Apple Music, Apple Pay, App Store/service/sync service problems.

PURCHASE_BILLING:
Payments, charges, subscriptions, purchases, orders, billing, preorder.

DEVICE_PERFORMANCE:
Slow device, lag, freezing, crashing, general performance problems.

BATTERY_CHARGING:
Battery drain, charging, overheating, power problems.

IOS_UPDATE_ISSUE:
iOS/software update failure, update bugs, update-related problems,
or downgrade/update issues.

HARDWARE_REPAIR:
Broken hardware, screen, physical buttons, warranty, repair, replacement.

NETWORK_CONNECTIVITY:
Wi-Fi, mobile network, Bluetooth, calling, internet connectivity.

FEATURE_SETTINGS:
Keyboard, camera, settings, UI behavior, app features, storage/settings.

CONTEXT_REQUIRED:
The current message is mainly a reaction or continuation and cannot
be understood properly without previous conversation.

UNKNOWN_ESCALATE:
The message is genuinely unclear and cannot reasonably be classified.
"""

def predict_intent_with_gemini(message):
    prompt = f"""
Classify the following customer-support tweet into EXACTLY ONE intent.

Customer message:
{message}

Allowed intents:
{taxonomy_text}

Rules:
- Choose the primary/root problem.
- Do not invent missing information.
- If the message is only a reaction such as "thanks", "yes",
  "okay", "that worked", classify as CONTEXT_REQUIRED.
- If an iOS update is explicitly the cause, prefer IOS_UPDATE_ISSUE.
- Use UNKNOWN_ESCALATE only when the message genuinely cannot be classified.

Return ONLY valid JSON:

{{
  "predicted_intent": "ONE_ALLOWED_INTENT",
  "confidence": 0.0,
  "reason": "brief reason"
}}
"""

    # Note: Use standard client.models.generate_content instead of non-existent client.interactions API
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={
            "temperature": 0.1,
            "response_mime_type": "application/json"
        }
    )

    result = json.loads(response.text.strip())
    intent = result["predicted_intent"]

    if intent not in VALID_INTENTS:
        intent = "UNKNOWN_ESCALATE"

    confidence = float(result.get("confidence", 0.0))
    confidence = max(0.0, min(1.0, confidence))

    return intent, confidence, result.get("reason", "")

    test_row = golden.iloc[0]

# We temporarily point MODEL_NAME to the verified active stable version
# to prevent the 503 spike error seen on the 3.8-flash endpoint.
MODEL_NAME = "gemini-3.6-flash"

try:
    intent, confidence, reason = predict_intent_with_gemini(test_row["text"])

    print("=== TEST OK ===")
    print("Customer:", test_row["text"])
    print("Human label:", test_row["final_intent"])
    print("Gemini prediction:", intent)
    print("Confidence:", round(confidence, 4))
    print("Reason:", reason)
except Exception as e:
    print(f"Test failed with: {type(e).__name__}: {e}")

In [ ]:
def generate_support_agent(message, tweet_id=None):

    # 1. Gemini classifies the intent
    predicted_intent, intent_confidence, intent_reason = (
        predict_intent_with_gemini(message)
    )

    # 2. Retrieve historical AppleSupport cases
    retrieved_cases = retrieve_similar_cases(
        message,
        top_k=5,
        exclude_tweet_id=tweet_id
    )

    # 3. Decide escalation
    decision, escalation_reason = decide_escalation(
        predicted_intent,
        intent_confidence,
        retrieved_cases
    )

    # 4. Format evidence
    evidence_text = format_evidence(retrieved_cases)

    # 5. Ask Gemini to write the response
    prompt = f"""
You are a customer-support response assistant.

CURRENT CUSTOMER MESSAGE:
{message}

PREDICTED INTENT:
{predicted_intent}

INTENT CONFIDENCE:
{intent_confidence:.4f}

INTENT REASON:
{intent_reason}

DECISION:
{decision}

ESCALATION REASON:
{escalation_reason}

HISTORICAL APPLESUPPORT EVIDENCE:
{evidence_text}

Write a concise, natural customer-support reply.

Rules:
- Use the historical examples as evidence.
- Do not copy historical replies word-for-word.
- Do not invent policies, refunds, warranties, guarantees, or facts.
- Do not mention tweet IDs or usernames from the historical cases.
- Do not pretend to be Apple.
- Do not include historical URLs.
- If the decision is ESCALATE, politely ask the customer to continue via DM.
- If the decision is AUTO_HANDLE, provide a useful low-risk response.
- Keep the reply concise.

Return ONLY JSON:

{{
    "draft_reply": "...",
    "evidence_used": [
        "short reason based on historical case 1",
        "short reason based on historical case 2"
    ]
}}
"""

    try:
        response = client.interactions.create(
            model=MODEL_NAME,
            input=prompt,
            generation_config={
                "thinking_level": "low"
            }
        )

        result = json.loads(response.output_text)

        return {
            "predicted_intent": predicted_intent,
            "intent_confidence": round(float(intent_confidence), 4),
            "draft_reply": result.get("draft_reply", ""),
            "decision": decision,
            "escalation_reason": escalation_reason,
            "evidence_used": result.get("evidence_used", [])
        }

    except Exception as e:

        return {
            "predicted_intent": predicted_intent,
            "intent_confidence": round(float(intent_confidence), 4),
            "draft_reply": (
                "We'd like to look into this with you. "
                "Please send us a direct message so we can gather more details."
            ),
            "decision": "ESCALATE",
            "escalation_reason": f"Reply generation failed: {type(e).__name__}",
            "evidence_used": []
        }

In [ ]:
test_row = golden.iloc[0]

agent_output = generate_support_agent(
    test_row["text"],
    tweet_id=test_row["tweet_id"]
)

print(json.dumps(agent_output, indent=2, ensure_ascii=False))

In [ ]:
from pydantic import BaseModel
from typing import List


class SupportReplyOutput(BaseModel):
    draft_reply: str
    evidence_used: List[str]


def generate_support_agent(message, tweet_id=None):

    # --------------------------------------------------------
    # 1. Predict intent with Gemini
    # --------------------------------------------------------

    predicted_intent, intent_confidence, intent_reason = (
        predict_intent_with_gemini(message)
    )

    # --------------------------------------------------------
    # 2. Retrieve historical cases
    # --------------------------------------------------------

    retrieved_cases = retrieve_similar_cases(
        message,
        top_k=5,
        exclude_tweet_id=tweet_id
    )

    # --------------------------------------------------------
    # 3. Decide AUTO_HANDLE vs ESCALATE
    # --------------------------------------------------------

    decision, escalation_reason = decide_escalation(
        predicted_intent,
        intent_confidence,
        retrieved_cases
    )

    # --------------------------------------------------------
    # 4. Format historical evidence
    # --------------------------------------------------------

    evidence_text = format_evidence(retrieved_cases)

    # --------------------------------------------------------
    # 5. Ask Gemini ONLY to generate the reply
    # --------------------------------------------------------

    prompt = f"""
You are a customer-support response assistant.

Current customer message:
{message}

Predicted intent:
{predicted_intent}

Intent confidence:
{intent_confidence:.4f}

Intent reasoning:
{intent_reason}

Final support decision:
{decision}

Escalation reason:
{escalation_reason}

Historical AppleSupport examples:
{evidence_text}

Write ONE concise, natural support reply.

Rules:
- Ground the response in the historical evidence.
- Do not copy historical replies word-for-word.
- Do not invent policies, refunds, warranties, guarantees, or unsupported facts.
- Do not include historical tweet IDs, usernames, or URLs.
- Do not mention that you are an AI.
- If the decision is ESCALATE, ask the customer to continue via DM.
- If the decision is AUTO_HANDLE, provide a useful low-risk response.
- Do not change the given decision.
- Keep the reply suitable for a Twitter/X support conversation.

Also provide 1–3 short statements explaining which historical evidence
supported the reply.
"""

    try:

        interaction = client.interactions.create(
            model=MODEL_NAME,
            input=prompt,
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": SupportReplyOutput.model_json_schema()
            }
        )

        # Structured JSON from Gemini
        reply_data = SupportReplyOutput.model_validate_json(
            interaction.output_text
        )

        return {
            "predicted_intent": predicted_intent,
            "intent_confidence": round(
                float(intent_confidence),
                4
            ),
            "draft_reply": reply_data.draft_reply.strip(),
            "decision": decision,
            "escalation_reason": escalation_reason,
            "evidence_used": reply_data.evidence_used
        }

    except Exception as e:

        # Safe fallback
        return {
            "predicted_intent": predicted_intent,
            "intent_confidence": round(
                float(intent_confidence),
                4
            ),
            "draft_reply": (
                "We'd like to look into this with you. "
                "Please send us a direct message so we can gather "
                "more details and continue troubleshooting."
            ),
            "decision": "ESCALATE",
            "escalation_reason": (
                f"Reply generation failed: {type(e).__name__}"
            ),
            "evidence_used": []
        }

In [ ]:
test_row = golden.iloc[0]

output = generate_support_agent(
    test_row["text"],
    tweet_id=test_row["tweet_id"]
)

print(json.dumps(
    output,
    indent=2,
    ensure_ascii=False
))

In [ ]:
def decide_escalation(
    predicted_intent,
    intent_confidence,
    retrieved_cases
):

    always_escalate = {
        "ACCOUNT_LOGIN",
        "PURCHASE_BILLING",
        "HARDWARE_REPAIR",
        "CONTEXT_REQUIRED",
        "UNKNOWN_ESCALATE",
        "IOS_UPDATE_ISSUE"
    }

    if predicted_intent in always_escalate:
        return (
            "ESCALATE",
            f"{predicted_intent} is better handled with human follow-up."
        )

    if retrieved_cases.empty:
        return (
            "ESCALATE",
            "No historical evidence was available."
        )

    top_similarity = float(
        retrieved_cases["similarity"].max()
    )

    if top_similarity < 0.20:
        return (
            "ESCALATE",
            "Historical evidence is too weak for safe automated handling."
        )

    if intent_confidence < 0.40:
        return (
            "ESCALATE",
            "Intent confidence is below the automation threshold."
        )

    return (
        "AUTO_HANDLE",
        "The issue is low-risk and supported by relevant historical examples."
    )

In [ ]:
import pandas as pd
import json
import time
import os

os.makedirs("results", exist_ok=True)

results = []

print("Starting FINAL evaluation on 200 human-reviewed golden examples...\n")

for i, row in golden.iterrows():

    try:
        output = generate_support_agent(
            message=row["text"],
            tweet_id=row["tweet_id"]
        )

        results.append({
            "tweet_id": row["tweet_id"],
            "text": row["text"],
            "actual_intent": row["final_intent"],
            "actual_multi_issue": row["final_multi_issue"],
            "actual_context_required": row["final_context_required"],

            "predicted_intent": output.get("predicted_intent"),
            "intent_confidence": output.get("intent_confidence"),

            "draft_reply": output.get("draft_reply"),
            "decision": output.get("decision"),
            "escalation_reason": output.get("escalation_reason"),

            "evidence_used": json.dumps(
                output.get("evidence_used", []),
                ensure_ascii=False
            )
        })

    except Exception as e:
        print(f"Error on row {i + 1}: {type(e).__name__}")

        results.append({
            "tweet_id": row["tweet_id"],
            "text": row["text"],
            "actual_intent": row["final_intent"],
            "actual_multi_issue": row["final_multi_issue"],
            "actual_context_required": row["final_context_required"],
            "predicted_intent": "UNKNOWN_ESCALATE",
            "intent_confidence": 0.0,
            "draft_reply": "",
            "decision": "ESCALATE",
            "escalation_reason": f"Agent error: {type(e).__name__}",
            "evidence_used": "[]"
        })

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/200")

    time.sleep(0.2)

evaluation_df = pd.DataFrame(results)

evaluation_df.to_csv(
    "results/golden_predictions.csv",
    index=False
)

print("\nFINAL EVALUATION COMPLETE")
print("Rows:", len(evaluation_df))
print("Saved: results/golden_predictions.csv")

In [ ]:
print("Total rows:", len(evaluation_df))

print("\nPredicted intents:")
print(evaluation_df["predicted_intent"].value_counts())

print("\nDecision:")
print(evaluation_df["decision"].value_counts())

print("\nErrors/fallback rows:")
print(
    evaluation_df[
        evaluation_df["escalation_reason"].astype(str).str.contains(
            "failed|error|fallback",
            case=False,
            na=False
        )
    ].shape[0]
)

In [ ]:
# FAST + RESUMABLE GOLDEN-SET EVALUATION
# 10 tweets per Gemini call

import os
import json
import time
import pandas as pd
from pydantic import BaseModel
from typing import List
from google import genai
from google.colab import userdata

# ------------------------------------------------------------
# 1. Gemini setup with robust API key loading
# ------------------------------------------------------------

try:
    # Try to load the standard secret name first
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    # Fall back directly to the raw key value instead of passing it to userdata.get()
    api_key = "REDACTED_API_KEY"

client = genai.Client(api_key=api_key)

# Use verified active model version
MODEL_NAME = "gemini-3.6-flash"

# ------------------------------------------------------------
# 2. Structured output schema
# ------------------------------------------------------------

class AgentItem(BaseModel):
    tweet_id: int
    predicted_intent: str
    intent_confidence: float
    draft_reply: str
    evidence_used: List[str]


class AgentBatchOutput(BaseModel):
    results: List[AgentItem]


VALID_INTENTS = [
    "ACCOUNT_LOGIN",
    "APPLE_SERVICE_ISSUE",
    "PURCHASE_BILLING",
    "DEVICE_PERFORMANCE",
    "BATTERY_CHARGING",
    "IOS_UPDATE_ISSUE",
    "HARDWARE_REPAIR",
    "NETWORK_CONNECTIVITY",
    "FEATURE_SETTINGS",
    "CONTEXT_REQUIRED",
    "UNKNOWN_ESCALATE"
]

# ------------------------------------------------------------
# 3. Local escalation policy
# ------------------------------------------------------------

def final_decision(predicted_intent, confidence, retrieved_cases):

    always_escalate = {
        "ACCOUNT_LOGIN",
        "PURCHASE_BILLING",
        "HARDWARE_REPAIR",
        "CONTEXT_REQUIRED",
        "UNKNOWN_ESCALATE",
        "IOS_UPDATE_ISSUE"
    }

    if predicted_intent in always_escalate:
        return (
            "ESCALATE",
            f"{predicted_intent} requires human follow-up."
        )

    if retrieved_cases.empty:
        return (
            "ESCALATE",
            "No historical AppleSupport evidence was found."
        )

    top_similarity = float(
        retrieved_cases["similarity"].max()
    )

    if top_similarity < 0.20:
        return (
            "ESCALATE",
            "Historical evidence is too weak for safe automation."
        )

    if confidence < 0.40:
        return (
            "ESCALATE",
            "Intent confidence is below the automation threshold."
        )

    return (
        "AUTO_HANDLE",
        "Low-risk issue with sufficient historical support evidence."
    )


# ------------------------------------------------------------
# 4. Build one batch prompt
# ------------------------------------------------------------

def make_batch_prompt(batch_rows):

    blocks = []

    for row in batch_rows:

        retrieved = retrieve_similar_cases(
            row["text"],
            top_k=3,
            exclude_tweet_id=row["tweet_id"]
        )

        evidence_lines = []

        for j, r in retrieved.reset_index(drop=True).iterrows():

            customer = str(r["customer_text"])[:500]
            support = str(r["support_reply"])[:500]

            evidence_lines.append(
                f"""
CASE {j+1}
Similarity: {float(r["similarity"]):.3f}
Customer: {customer}
AppleSupport: {support}
""".strip()
            )

        evidence = "\n\n".join(evidence_lines)

        blocks.append(
            f"""
TWEET_ID: {int(row["tweet_id"])}

CUSTOMER MESSAGE:
{row["text"]}

HISTORICAL EVIDENCE:
{evidence}
""".strip()
        )

    all_examples = "\n\n============================\n\n".join(blocks)

    prompt = f"""
You are an AI customer-support agent for a research prototype
based on historical AppleSupport Twitter conversations.

Classify and draft a response for EACH customer message below.

ALLOWED INTENTS:
{', '.join(VALID_INTENTS)}

INTENT DEFINITIONS:

ACCOUNT_LOGIN:
Apple ID, iCloud login, passwords, sign-in, activation lock, account access.

APPLE_SERVICE_ISSUE:
Apple Music, Apple Pay, App Store/service/sync problems.

PURCHASE_BILLING:
Payments, charges, subscriptions, purchases, orders, billing.

DEVICE_PERFORMANCE:
Slow device, lag, freezing, crashing, general performance.

BATTERY_CHARGING:
Battery drain, charging, overheating, power issues.

IOS_UPDATE_ISSUE:
Problems explicitly caused by iOS/software updates, update failures,
update bugs, or downgrade/update problems.

HARDWARE_REPAIR:
Broken hardware, screen, buttons, physical damage, warranty, repair,
replacement.

NETWORK_CONNECTIVITY:
Wi-Fi, mobile network, Bluetooth, calling, internet connectivity.

FEATURE_SETTINGS:
Keyboard, camera, settings, UI behavior, app features, storage/settings.

CONTEXT_REQUIRED:
The current message cannot be understood properly without previous
conversation context. Examples include "thanks", "yes", "okay",
"that worked", incomplete reactions, etc.

UNKNOWN_ESCALATE:
Genuinely unclear messages that cannot reasonably be classified.

RULES:

1. Choose exactly ONE primary intent.
2. Do not invent missing information.
3. If the customer explicitly blames an iOS update, prefer IOS_UPDATE_ISSUE.
4. Use CONTEXT_REQUIRED only when the current message really needs
   previous conversation context.
5. Use the historical cases as evidence, NOT as text to copy.
6. Create a NEW reply.
7. Never include historical tweet IDs, usernames, or URLs.
8. Do not invent Apple policies, refunds, warranties, guarantees,
   or unsupported troubleshooting instructions.
9. Keep replies concise and natural.
10. If historical evidence mostly shows AppleSupport moving similar
    cases to DM, the reply can appropriately ask the customer to
    continue via DM.
11. Confidence must be between 0.0 and 1.0.
12. Return one result for EVERY TWEET_ID.

CUSTOMER CASES:

{all_examples}

Return ONLY structured JSON matching the required schema.
"""

    return prompt


# ------------------------------------------------------------
# 5. Process one batch
# ------------------------------------------------------------

def process_batch(batch_rows, max_retries=3):

    prompt = make_batch_prompt(batch_rows)

    for attempt in range(max_retries):

        try:
            # Use the correct client.models.generate_content structured output mechanism
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": AgentBatchOutput,
                    "temperature": 0.1
                }
            )

            parsed = AgentBatchOutput.model_validate_json(
                response.text.strip()
            )

            return parsed.results

        except Exception as e:

            print(
                f"Batch attempt {attempt + 1} failed: "
                f"{type(e).__name__}: {str(e)[:250]}"
            )

            if attempt < max_retries - 1:

                wait_time = 2 ** attempt * 3

                print(
                    f"Waiting {wait_time} seconds before retry..."
                )

                time.sleep(wait_time)

    raise RuntimeError(
        "Batch failed after all retries."
    )


# ------------------------------------------------------------
# 6. Resume from existing progress
# ------------------------------------------------------------

output_file = "results/golden_predictions_batch.csv"

os.makedirs("results", exist_ok=True)

if os.path.exists(output_file):

    existing = pd.read_csv(output_file)

    completed_ids = set(
        existing["tweet_id"].astype(str)
    )

    saved_results = existing.to_dict("records")

    print(
        f"Resuming evaluation. "
        f"{len(completed_ids)} tweets already completed."
    )

else:

    completed_ids = set()
    saved_results = []

    print("Starting new evaluation.")


# ------------------------------------------------------------
# 7. Select remaining examples
# ------------------------------------------------------------

remaining = golden[
    ~golden["tweet_id"].astype(str).isin(completed_ids)
].copy()

print(
    f"Remaining examples: {len(remaining)} / {len(golden)}"
)


# ------------------------------------------------------------
# 8. Run in batches of 10
# ------------------------------------------------------------

BATCH_SIZE = 10

for start in range(0, len(remaining), BATCH_SIZE):

    batch = remaining.iloc[
        start:start + BATCH_SIZE
    ]

    print(
        f"\nProcessing batch "
        f"{start + 1}-{start + len(batch)} "
        f"of {len(remaining)}..."
    )

    batch_outputs = process_batch(
        batch.to_dict("records")
    )

    # Map Gemini results by tweet ID
    output_map = {
        str(x.tweet_id): x
        for x in batch_outputs
    }

    for _, row in batch.iterrows():

        tweet_id = str(row["tweet_id"])

        if tweet_id not in output_map:

            print(
                f"WARNING: Gemini did not return tweet "
                f"{tweet_id}"
            )

            continue

        result = output_map[tweet_id]

        predicted_intent = result.predicted_intent

        if predicted_intent not in VALID_INTENTS:
            predicted_intent = "UNKNOWN_ESCALATE"

        confidence = max(
            0.0,
            min(
                1.0,
                float(result.intent_confidence)
            )
        )

        # Retrieve again for local escalation policy
        retrieved = retrieve_similar_cases(
            row["text"],
            top_k=3,
            exclude_tweet_id=row["tweet_id"]
        )

        decision, escalation_reason = final_decision(
            predicted_intent,
            confidence,
            retrieved
        )

        saved_results.append({

            "tweet_id": row["tweet_id"],

            "text": row["text"],

            "actual_intent": row["final_intent"],

            "actual_multi_issue": row["final_multi_issue"],

            "actual_context_required":
                row["final_context_required"],

            "predicted_intent":
                predicted_intent,

            "intent_confidence":
                round(confidence, 4),

            "draft_reply":
                result.draft_reply.strip(),

            "decision":
                decision,

            "escalation_reason":
                escalation_reason,

            "evidence_used":
                json.dumps(
                    result.evidence_used,
                    ensure_ascii=False
                )
        })

    # --------------------------------------------------------
    # SAVE AFTER EVERY BATCH
    # --------------------------------------------------------

    temp_df = pd.DataFrame(saved_results)

    temp_df = temp_df.drop_duplicates(
        subset=["tweet_id"],
        keep="last"
    )

    temp_df.to_csv(
        output_file,
        index=False
    )

    print(
        f"Saved progress: "
        f"{len(temp_df)}/200"
    )

    # Small pause between batches to respect rate limits
    time.sleep(3)


# ------------------------------------------------------------
# 9. Finalize
# ------------------------------------------------------------

evaluation_df = pd.read_csv(output_file)

evaluation_df = evaluation_df.drop_duplicates(
    subset=["tweet_id"]
)

evaluation_df.to_csv(
    "results/golden_predictions.csv",
    index=False
)

print("\n====================================")
print("FINAL EVALUATION FINISHED")
print("====================================")
print("Rows:", len(evaluation_df))
print("File:", "results/golden_predictions.csv")